# 🚀 머신러닝 실습 : 고객 구매 데이터로 성별 예측 모델링 (분류 문제)

* 주어진 데이터는 백화점 고객의 1년 간 구매 데이터입니다.
* 고객 3,500명에 대한 학습용 데이터(y.csv, X.csv)를 이용하여 성별예측 모형을 만들어보세요.
* 모델의 성능은 자유롭게 측정해봅니다!

## [실습 프로세스]
1. 데이터 불러오기  
2. 데이터 탐색
3. 데이터 전처리  
4. 학습/테스트 데이터 분리  
5. 모델 선택 및 학습  
6. 예측 및 평가  


<br/>

---

<br/>
<br/>

# 0. 라이브러리 불러오기

* 라이브러리를 가져와서 과정을 준비합니다

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings(action='ignore')

<br/>

---

<br/>
<br/>

# 1. 데이터 불러오기
* 데이터를 가져와서 과정을 준비합시다.
- 인코딩 방식은 'euc-kr' 을 활용하세요.
- 데이터 출처 : 한국데이터산업진흥원 빅데이터분석기사 실기 공개 예시 문항

- 독립 변수 데이터셋 : ./data/X.csv
- 종속 변수 데이터셋 : ./data/y.csv


데이터 파일을 불러옵니다. 보통 CSV 파일을 pandas로 읽어옵니다.

In [5]:
import os
# 노트북 파일이 있는 폴더로 이동 (예시)
os.chdir(r'C:\githome\hipython_rep')

# 변경 후 확인
print("변경 후:", os.getcwd())

변경 후: C:\githome\hipython_rep


In [6]:
X = pd.read_csv('./data1/X.csv', encoding='euc-kr')
y = pd.read_csv('./data1/y.csv', encoding='euc-kr')

<br/>

---

<br/>
<br/>

# 2. 데이터 탐색하기
* 데이터를 이해할 수 있도록 탐색과정을 수행해봅시다.


### 데이터의 상위 몇 개 행을 출력하여 전체 구조를 미리 확인합니다.

In [8]:
X.head()

,cust_id,총구매액,최대구매액,환불금액,주구매상품,주구매지점,내점일수,내점당구매건수,주말방문비율,구매주기
0,0,68282840,11264000,6860000.0,기타,강남점,19,3.894737,0.527027,17
1,1,2136000,2136000,300000.0,스포츠,잠실점,2,1.500000,0.000000,1
2,2,3197000,1639000,NaN,남성 캐주얼,관악점,2,2.000000,0.000000,1
3,3,16077620,4935000,NaN,기타,광주점,18,2.444444,0.318182,16
4,4,29050000,24000000,NaN,보석,본 점,2,1.500000,0.000000,85


In [9]:
y.head()

,cust_id,gender
0,0,0
1,1,0
2,2,1
3,3,1
4,4,0



### 데이터의 요약 정보나 통계 정보를 출력해 변수들의 유형과 분포를 확인합니다.

In [ ]:
X.info()   # X 데이터 요약 정보

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3500 entries, 0 to 3499
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   cust_id  3500 non-null   int64  
 1   총구매액     3500 non-null   int64  
 2   최대구매액    3500 non-null   int64  
 3   환불금액     1205 non-null   float64
 4   주구매상품    3500 non-null   object 
 5   주구매지점    3500 non-null   object 
 6   내점일수     3500 non-null   int64  
 7   내점당구매건수  3500 non-null   float64
 8   주말방문비율   3500 non-null   float64
 9   구매주기     3500 non-null   int64  
dtypes: float64(3), int64(5), object(2)
memory usage: 273.6+ KB


+ X 데이터에는 '환불금액'에 결측치가 2295개가 있다.

In [ ]:
y.info()   # y 데이터 요약 정보

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3500 entries, 0 to 3499
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   cust_id  3500 non-null   int64
 1   gender   3500 non-null   int64
dtypes: int64(2)
memory usage: 54.8 KB


In [ ]:
X.describe()   # X 데이터 통계 정보

,cust_id,총구매액,최대구매액,환불금액,내점일수,내점당구매건수,주말방문비율,구매주기
count,3500.000000,3.500000e+03,3.500000e+03,1.205000e+03,3500.000000,3500.000000,3500.000000,3500.000000
mean,1749.500000,9.191925e+07,1.966424e+07,2.407822e+07,19.253714,2.834963,0.307246,20.958286
std,1010.507298,1.635065e+08,3.199235e+07,4.746453e+07,27.174942,1.912368,0.289752,24.748682
min,0.000000,-5.242152e+07,-2.992000e+06,5.600000e+03,1.000000,1.000000,0.000000,0.000000
25%,874.750000,4.747050e+06,2.875000e+06,2.259000e+06,2.000000,1.666667,0.027291,4.000000
50%,1749.500000,2.822270e+07,9.837000e+06,7.392000e+06,8.000000,2.333333,0.256410,13.000000
75%,2624.250000,1.065079e+08,2.296250e+07,2.412000e+07,25.000000,3.375000,0.448980,28.000000
max,3499.000000,2.323180e+09,7.066290e+08,5.637530e+08,285.000000,22.083333,1.000000,166.000000


+ 주말방문비율의 평균이 약 30.7%이므로 고객들이 주로 주말보다 평일에 더 많이 구매한다고 판단할 수 있다.
+ 구매주기의 평균이 약 21일이므로 보통 고객들은 약 3주마다 구매한다고 판단하여 이 기간마다 프로모션을 진행해봐도 좋을 듯 하다.

In [ ]:
y.describe()   # y 데이터 통계 정보

,cust_id,gender
count,3500.000000,3500.000000
mean,1749.500000,0.376000
std,1010.507298,0.484449
min,0.000000,0.000000
25%,874.750000,0.000000
50%,1749.500000,0.000000
75%,2624.250000,1.000000
max,3499.000000,1.000000


+ 여자(gender=0), 남자(gender=1)의 통계를 보면 평균값(mean)과 표준편차값(std)이 50% 이하이므로 해당 데이터에는 남성보다 여성이 더 많다고 분류할 수 있다.

<br/>

---

<br/>
<br/>

# 3. 데이터 전처리
* 전처리 과정을 통해서 머신러닝에 사용할 수 있는 형태의 데이터 준비


### 필요한 라이브러리를 불러옵니다.
- 인코딩 : LabelEncoder
- 데이터 표준화 : StandardScaler

In [14]:
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

### 단순히 1부터의 숫자를 부여한 'cust_id'를 수치형 변수로 받아들이면, 결과가 왜곡될 수 있으니 컬럼을 제거합니다.

In [16]:
X_new = X.copy()
X_new = X_new.drop('cust_id', axis=1)
X_new

,총구매액,최대구매액,환불금액,주구매상품,주구매지점,내점일수,내점당구매건수,주말방문비율,구매주기
0,68282840,11264000,6860000.0,기타,강남점,19,3.894737,0.527027,17
1,2136000,2136000,300000.0,스포츠,잠실점,2,1.500000,0.000000,1
2,3197000,1639000,NaN,남성 캐주얼,관악점,2,2.000000,0.000000,1
3,16077620,4935000,NaN,기타,광주점,18,2.444444,0.318182,16
4,29050000,24000000,NaN,보석,본 점,2,1.500000,0.000000,85
...,...,...,...,...,...,...,...,...,...
3495,3175200,3042900,NaN,골프,본 점,1,2.000000,1.000000,0
3496,29628600,7200000,6049600.0,시티웨어,부산본점,8,1.625000,0.461538,40
3497,75000,75000,NaN,주방용품,창원점,1,1.000000,0.000000,0
3498,1875000,1000000,NaN,화장품,본 점,2,1.000000,0.000000,39


### 데이터에 결측치가 있는지 확인해보세요


In [18]:
X_new.isnull().sum()

총구매액          0
최대구매액         0
환불금액       2295
주구매상품         0
주구매지점         0
내점일수          0
내점당구매건수       0
주말방문비율        0
구매주기          0
dtype: int64

+ '환불금액' 컬럼에 결측치가 2295개 존재한다.

### 결측치에 0으로 채워 넣어 모델 학습에 지장이 없도록 합니다.

In [21]:
X_neww = X_new.fillna(0)
X_neww.isnull().sum()

총구매액       0
최대구매액      0
환불금액       0
주구매상품      0
주구매지점      0
내점일수       0
내점당구매건수    0
주말방문비율     0
구매주기       0
dtype: int64

In [23]:
X_neww.head()

,총구매액,최대구매액,환불금액,주구매상품,주구매지점,내점일수,내점당구매건수,주말방문비율,구매주기
0,68282840,11264000,6860000.0,기타,강남점,19,3.894737,0.527027,17
1,2136000,2136000,300000.0,스포츠,잠실점,2,1.500000,0.000000,1
2,3197000,1639000,0.0,남성 캐주얼,관악점,2,2.000000,0.000000,1
3,16077620,4935000,0.0,기타,광주점,18,2.444444,0.318182,16
4,29050000,24000000,0.0,보석,본 점,2,1.500000,0.000000,85



### 문자형 범주 데이터를 숫자로 바꾸기 위한 인코딩을 수행합니다.

In [28]:
X_le = LabelEncoder()
col = ['주구매상품', '주구매지점']   # 범주형 데이터 컬럼 두 개

for i in col:
    if i in X_neww.columns:
        X_neww[i] = X_le.fit_transform(X_neww[i])   # 레이블 인코딩
        
X_neww.head()

,총구매액,최대구매액,환불금액,주구매상품,주구매지점,내점일수,내점당구매건수,주말방문비율,구매주기
0,68282840,11264000,6860000.0,5,0,19,3.894737,0.527027,17
1,2136000,2136000,300000.0,21,19,2,1.500000,0.000000,1
2,3197000,1639000,0.0,6,1,2,2.000000,0.000000,1
3,16077620,4935000,0.0,5,2,18,2.444444,0.318182,16
4,29050000,24000000,0.0,15,8,2,1.500000,0.000000,85


### 각 데이터에 표준화를 적용하여 데이터의 스케일(크기 차이)을 맞춰줍니다.
- 평균을 0, 표준편차를 1로 맞춰서 → 데이터가 정규 분포 형태로 변환되도록 하세요

In [29]:
scalar = StandardScaler()
scalar.fit(X_neww)
X_scaled = scalar.transform(X_neww)
X_scaled.mean()

np.float64(-4.1504718533288394e-17)

In [30]:
X_scaled.var()

np.float64(1.0)

<br/>

---

<br/>
<br/>

# 5-1. 모델링 - LogisticRegression

* 본격적으로 모델을 선언하고 학습시킵니다.


### 필요한 라이브러리를 불러옵니다.

In [241]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

x_1 = X_neww[['총구매액', '최대구매액','환불금액', '주구매상품', '주구매지점', '내점일수', '내점당구매건수', '주말방문비율', '구매주기']]
y_1 = y[['gender']]

X_train, X_test, y_train, y_test = train_test_split(x_1,
                                                    y_1,
                                                    test_size=0.2,
                                                    random_state=11)

### 모델을 선언하여 객체화시킵니다.

In [242]:
# 스케일링 하지 않았을 때
no_scaling_model = LogisticRegression(random_state=11)
no_scaling_model.fit(X_train, y_train)

LogisticRegression(random_state=11)

In [243]:
# 스케일링
from sklearn.preprocessing import StandardScaler
scaler1 = StandardScaler()
scaler1.fit(X_train)
X_train_scaled = scaler1.transform(X_train)
X_test_scaled = scaler1.transform(X_test)

In [244]:
# 스케일링 했을 때
scaling_model = LogisticRegression(random_state=11)
scaling_model.fit(X_train_scaled, y_train)

LogisticRegression(random_state=11)


### 모델을 학습 데이터에 맞춰 학습시킵니다.

In [245]:
# 스케일링 하지 않았을 때
pred_1 = no_scaling_model.predict(X_test)

In [246]:
# 스케일링 했을 때
pred_2 = scaling_model.predict(X_test_scaled)

<br/>

---

<br/>
<br/>

# 6-1. 예측 성능 확인해보기 - LogisticRegression

- 학습된 모델로 테스트 데이터에 대한 예측을 수행합니다.

- 학습시킨 모델의 성능을 알아봅니다
- 각 평가지표로 모델의 성능을 수치화하여 확인합니다.
- 필요한 라이브러리를 import 하고 성능을 확인해보세요 (정확도, 정밀도, 재현율, f1, confusion_matrix)

In [247]:
# 라이브러리 import
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import precision_score, recall_score

### 정확도 - LogisticRegression

In [248]:
acc_1 = accuracy_score(y_test, pred_1)   # 스케일링 안했을 때
acc_2 = accuracy_score(y_test, pred_2)   # 스케일링 했을 때
acc_2, acc_1

(0.6242857142857143, 0.6271428571428571)

+ 스케일링을 하지 않았을 때는 정확도가 약 62.4%이다.
+ 스케일링을 했을 때는 정확도가 약 62.7%이다.

### 정밀도 - LogisticRegression

In [249]:
#pred_1 = no_scaling_model.predict(X_test)   # 스케일링 안했을 때

#pred_2 = scaling_model.predict(X_test)   # 스케일링 했을 때
precision_score(y_test, pred_1), precision_score(y_test, pred_2)

(np.float64(0.0), np.float64(0.48214285714285715))

+ 스케일링 후 정밀도는 약 48.2%으로 나온다. 

### 재현율 - LogisticRegression

In [250]:
recall_score(y_test, pred_1), recall_score(y_test, pred_2)

(np.float64(0.0), np.float64(0.10344827586206896))

+ 스케일링 후 재현율은 약 10.3%으로 나온다. 

### F1 score - LogisticRegression

In [251]:
from sklearn.metrics import f1_score
f1_score(y_test, pred_1), f1_score(y_test, pred_2)

(np.float64(0.0), np.float64(0.17034700315457413))

+ 스케일링 후 F1 score는 약 0.17으로 나온다. 

### 혼동행렬 - LogisticRegression

In [252]:
from sklearn.metrics import confusion_matrix

confusion_matrix(y_test, pred_1)   # 스케일링 안했을 때

array([[439,   0],
       [261,   0]])

In [253]:
confusion_matrix(y_test, pred_1)   # 스케일링 안했을 때

array([[439,   0],
       [261,   0]])

+ 스케일링 유무에 관계없이 혼동행렬이 같게 나온다.
+ 남자라고 예측한 값이 0이므로 이 모델은 여자라고만 예측하기 때문에 편향적이다.

### 평가보고서 - LogisticRegression

In [254]:
from sklearn.metrics import classification_report

print(classification_report(y_test, pred_1))

              precision    recall  f1-score   support

           0       0.63      1.00      0.77       439
           1       0.00      0.00      0.00       261

    accuracy                           0.63       700
   macro avg       0.31      0.50      0.39       700
weighted avg       0.39      0.63      0.48       700




<br/>

---

<br/>
<br/>

# 5-2. 모델링 - DecisionTreeClassifier

* 본격적으로 모델을 선언하고 학습시킵니다.


### 필요한 라이브러리를 불러옵니다.

In [255]:
from sklearn.tree import DecisionTreeClassifier

### 모델을 선언하여 객체화시킵니다.

In [256]:
tree_model = DecisionTreeClassifier(random_state=11)

### 모델을 학습 데이터에 맞춰 학습시킵니다.

In [257]:
tree_model.fit(X_train, y_train)

DecisionTreeClassifier(random_state=11)

In [258]:
tree_pred = tree_model.predict(X_test)



<br/>
<br/>

# 6-2. 예측 성능 확인해보기 - DecisionTreeClassifier

- 학습된 모델로 테스트 데이터에 대한 예측을 수행합니다.

- 학습시킨 모델의 성능을 알아봅니다
- 각 평가지표로 모델의 성능을 수치화하여 확인합니다.
- 필요한 라이브러리를 import 하고 성능을 확인해보세요 (정확도, 정밀도, 재현율, f1, confusion_matrix)

### 정확도 - DecisionTreeClassifier

In [259]:
accuracy_score(y_test, tree_pred)

0.5614285714285714

+ 정확도는 약 56.1%이다.

### 정밀도 - DecisionTreeClassifier

In [260]:
precision_score(y_test, tree_pred)

np.float64(0.4148148148148148)

+ 정밀도는 약 41.5%이다.

### 재현율 - DecisionTreeClassifier

In [261]:
recall_score(y_test, tree_pred)

np.float64(0.42911877394636017)

+ 재현율은 약 42.9%이다.

### F1 score - DecisionTreeClassifier

In [262]:
f1_score(y_test, tree_pred)

np.float64(0.4218455743879473)

+ F1 score는 약 0.422이다.

### 혼동행렬 - DecisionTreeClassifier

In [263]:
confusion_matrix(y_test, tree_pred)

array([[281, 158],
       [149, 112]])

+ 남자라고 예측했는데 실제 남자인 수치가 가장 낮게 나온다.
+ 정확도, 정밀도, 재현율, F1 score를 보면 이 모델의 성능이 그리 좋다고 말할 수 없다.

### 평가보고서 - DecisionTreeClassifier

In [264]:
print(classification_report(y_test, tree_pred))

              precision    recall  f1-score   support

           0       0.65      0.64      0.65       439
           1       0.41      0.43      0.42       261

    accuracy                           0.56       700
   macro avg       0.53      0.53      0.53       700
weighted avg       0.56      0.56      0.56       700




<br/>

---

<br/>
<br/>

# 5-3. 모델링 - RandomForestClassifier

* 본격적으로 모델을 선언하고 학습시킵니다.



### 필요한 라이브러리를 불러옵니다.

In [265]:
from sklearn.ensemble import RandomForestClassifier

### 모델을 선언하여 객체화시킵니다.

In [266]:
rf_model = RandomForestClassifier(random_state=11)

### 모델을 학습 데이터에 맞춰 학습시킵니다.

In [267]:
rf_model.fit(X_train, y_train)

RandomForestClassifier(random_state=11)

In [268]:
pred_rf = rf_model.predict(X_test)



<br/>
<br/>

# 6-3. 예측 성능 확인해보기 - RandomForestClassifier

- 학습된 모델로 테스트 데이터에 대한 예측을 수행합니다.

- 학습시킨 모델의 성능을 알아봅니다
- 각 평가지표로 모델의 성능을 수치화하여 확인합니다.
- 필요한 라이브러리를 import 하고 성능을 확인해보세요 (정확도, 정밀도, 재현율, f1, confusion_matrix)

### 정확도 - RandomForestClassifier

In [269]:
accuracy_score(y_test, pred_rf)

0.6185714285714285

+ 정확도는 약 61.9%이다.

### 정밀도 - RandomForestClassifier

In [270]:
precision_score(y_test, pred_rf)

np.float64(0.4805194805194805)

+ 정밀도는 약 48.1%이다.

### 재현율 - RandomForestClassifier

In [271]:
recall_score(y_test, pred_rf)

np.float64(0.2835249042145594)

+ 재현율은 약 28.4%이다.

### F1 score - RandomForestClassifier

In [272]:
f1_score(y_test, pred_rf)

np.float64(0.3566265060240964)

+ F1 score는 약 0.357이다.

### 혼동행렬 - RandomForestClassifier

In [273]:
confusion_matrix(y_test, pred_rf)

array([[359,  80],
       [187,  74]])

+ 남자라고 예측했는데 실제 남자인 수치가 가장 낮게 나온다.
+ 정확도만 보면 타 모델에 비해 높지만, 다른 평가 지표도 보았을 때 모델의 성능이 그리 좋다고 하기 어렵다.

### 평가보고서 - RandomForestClassifier

In [274]:
print(classification_report(y_test, pred_rf))

              precision    recall  f1-score   support

           0       0.66      0.82      0.73       439
           1       0.48      0.28      0.36       261

    accuracy                           0.62       700
   macro avg       0.57      0.55      0.54       700
weighted avg       0.59      0.62      0.59       700




<br/>

---

<br/>
<br/>

# 5-4. 모델링 - XGBoost

* 본격적으로 모델을 선언하고 학습시킵니다.



### 필요한 라이브러리를 불러옵니다.

In [275]:
from xgboost import XGBClassifier

### 모델을 선언하여 객체화시킵니다.

In [276]:
xgb_model = XGBClassifier(random_state=11, use_label_encoder=False, eval_metric='logloss')

### 모델을 학습 데이터에 맞춰 학습시킵니다.

In [277]:
xgb_model.fit(X_train, y_train)

XGBClassifier(base_score=0.5, booster='gbtree', callbacks=None,
              colsample_bylevel=1, colsample_bynode=1, colsample_bytree=1,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric='logloss', gamma=0, gpu_id=-1,
              grow_policy='depthwise', importance_type=None,
              interaction_constraints='', learning_rate=0.300000012,
              max_bin=256, max_cat_to_onehot=4, max_delta_step=0, max_depth=6,
              max_leaves=0, min_child_weight=1, missing=nan,
              monotone_constraints='()', n_estimators=100, n_jobs=0,
              num_parallel_tree=1, predictor='auto', random_state=11,
              reg_alpha=0, reg_lambda=1, ...)

In [278]:
pred_xgb = xgb_model.predict(X_test)



<br/>
<br/>

# 6-4. 예측 성능 확인해보기 - XGBoost

- 학습된 모델로 테스트 데이터에 대한 예측을 수행합니다.

- 학습시킨 모델의 성능을 알아봅니다
- 각 평가지표로 모델의 성능을 수치화하여 확인합니다.
- 필요한 라이브러리를 import 하고 성능을 확인해보세요 (정확도, 정밀도, 재현율, f1, confusion_matrix)

### 정확도 - XGBoost

In [279]:
accuracy_score(y_test, pred_xgb)

0.6285714285714286

+ 정확도는 약 62.9%이다.

### 정밀도 - XGBoost

In [280]:
precision_score(y_test, pred_xgb)

np.float64(0.5024154589371981)

+ 정밀도는 약 50.2%이다.

### 재현율 - XGBoost

In [281]:
recall_score(y_test, pred_xgb)

np.float64(0.39846743295019155)

+ 재현율은 약 39.8%이다.

### F1 score - XGBoost

In [282]:
f1_score(y_test, pred_xgb)

np.float64(0.4444444444444444)

+ F1 score는 약 0.444이다.

### 혼동행렬 - XGBoost

In [283]:
confusion_matrix(y_test, pred_xgb)

array([[336, 103],
       [157, 104]])

+ 남자라고 예측했는데 실제로는 남자인지, 여자인지 수치가 반반(정밀도  50%)이 나온다.
+ 정확도, 정밀도, F1 score만 보면 타 모델에 비해 가장 높게 측정된다.

### 평가보고서 - XGBoost

In [284]:
print(classification_report(y_test, pred_xgb))

              precision    recall  f1-score   support

           0       0.68      0.77      0.72       439
           1       0.50      0.40      0.44       261

    accuracy                           0.63       700
   macro avg       0.59      0.58      0.58       700
weighted avg       0.61      0.63      0.62       700



<br/>

---


<br/>

## 7.  위 4가지 모델의 학습 & 예측 & 평가 결과를 확인하고 최고 성능을 내는 모델을 찾아봅시다!

- 어떤 모델이 가장 성능이 좋은가요 ?

4가지 모델의 평가 지표를 비교해본 결과, **XGBoost 모델**이 가장 높은 정확도(62.9%)와 정밀도(50.2%)와 F1 Score(0.444)를 기록하여 전반적으로 가장 우수한 성능을 보였습니다.<br>
 재현율(39.8%) 측면에서도 다른 모델에 비해 균형이 잘 잡혀 있었으며, 특히 실제 남성을 잘 분류해내는 데 강점을 보였습니다.

 다만, 전체적으로 성능 수치가 매우 높다고 보기는 어려우며, 아직 개선의 여지가 있습니다.<br> 예를 들어, 하이퍼파라미터 튜닝, 교차 검증을 통한 최적 모델 탐색, 더 많은 피처(변수) 추가 또는 언더샘플링/오버샘플링과 같은 불균형 데이터 처리를 통해 성능을 더욱 향상시킬 수 있습니다. 